In [16]:
!pip install -q langchain-core requests langchain-ollama

In [17]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [18]:
llm = ChatOllama(
    model="qwen3:4b",
    temperature=0
)

## tool create : 

In [19]:
@tool
def multiply(a:int , b:int)->int:
    """ Given 2 numbers a and b this tool returns their product"""
    return a * b

In [20]:
print(multiply.invoke({'a' : 3 , 'b' : 4}))

12


In [21]:
print(multiply.name)
print(multiply.description)
print(multiply.args)


multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## tool binding : 

In [22]:
llm_with_tools = llm.bind_tools([multiply])
llm_with_tools

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, model='qwen3:4b', temperature=0.0), kwargs={'tools': [{'type': 'function', 'function': {'name': 'multiply', 'description': 'Given 2 numbers a and b this tool returns their product', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

## tool Calling : 

In [23]:
llm_with_tools.invoke('hi how are you?')

AIMessage(content="The user is greeting me, but there are no relevant functions available to respond to this query. I don't have a tool for handling greetings or status checks. I'll just respond politely without using any tools.\n\n<response>\nHi there! I'm doing well, thank you! How can I assist you today?\n</response>", additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-19T17:15:09.2475534Z', 'done': True, 'done_reason': 'stop', 'total_duration': 19743495200, 'load_duration': 5463329800, 'prompt_eval_count': 147, 'prompt_eval_duration': 1853249000, 'eval_count': 172, 'eval_duration': 12418006000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0baa9-dd1d-7792-952b-63dc727103e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 147, 'output_tokens': 172, 'total_tokens': 319})

In [24]:
llm_with_tools.invoke('Can you multiply 2 with 10?')

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-19T17:18:13.4553858Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13208518800, 'load_duration': 20905800, 'prompt_eval_count': 152, 'prompt_eval_duration': 313358000, 'eval_count': 176, 'eval_duration': 12838072000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0baac-c634-7163-8ce5-02b8ae56024a-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 10}, 'id': '78ca823f-c6c4-43c6-a86a-7ab8572d0797', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 176, 'total_tokens': 328})

In [35]:
query = HumanMessage('can you multiply 2 with 10?')

In [36]:
messages = [query]

In [37]:
messages

[HumanMessage(content='can you multiply 2 with 10?', additional_kwargs={}, response_metadata={})]

In [44]:
result = llm_with_tools.invoke(messages)
result

AIMessage(content='The result of multiplying 2 by 10 is 20.', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-19T17:36:36.6655994Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20607778800, 'load_duration': 4870100, 'prompt_eval_count': 190, 'prompt_eval_duration': 88861000, 'eval_count': 268, 'eval_duration': 20480901000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0babd-7eb8-7f83-8b94-f8925201f4be-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 190, 'output_tokens': 268, 'total_tokens': 458})

In [38]:
messages.append(result)

In [39]:
messages

[HumanMessage(content='can you multiply 2 with 10?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-19T17:27:21.5249095Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18377110700, 'load_duration': 24670100, 'prompt_eval_count': 152, 'prompt_eval_duration': 79009000, 'eval_count': 241, 'eval_duration': 18215475000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0bab5-0ee5-7fa2-b316-52a0b60319d2-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 10}, 'id': '4b8565df-94c0-4767-a937-9e914a273674', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 241, 'total_tokens': 393})]

In [32]:
result.tool_calls[0]


{'name': 'multiply',
 'args': {'a': 2, 'b': 10},
 'id': '4b8565df-94c0-4767-a937-9e914a273674',
 'type': 'tool_call'}

In [30]:
result.tool_calls[0]['args']

{'a': 2, 'b': 10}

## tool Execution : 

In [31]:
multiply.invoke(result.tool_calls[0]['args'])

20

In [33]:
multiply.invoke({'name': 'multiply',
 'args': {'a': 2, 'b': 10},
 'id': '4b8565df-94c0-4767-a937-9e914a273674',
 'type': 'tool_call'})

ToolMessage(content='20', name='multiply', tool_call_id='4b8565df-94c0-4767-a937-9e914a273674')

In [40]:
tool_result = multiply.invoke(result.tool_calls[0])

In [41]:
messages.append(tool_result)

In [42]:
messages

[HumanMessage(content='can you multiply 2 with 10?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-09-19T17:27:21.5249095Z', 'done': True, 'done_reason': 'stop', 'total_duration': 18377110700, 'load_duration': 24670100, 'prompt_eval_count': 152, 'prompt_eval_duration': 79009000, 'eval_count': 241, 'eval_duration': 18215475000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a0bab5-0ee5-7fa2-b316-52a0b60319d2-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 10}, 'id': '4b8565df-94c0-4767-a937-9e914a273674', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 241, 'total_tokens': 393}),
 ToolMessage(content='20', name='multiply', tool_call_id='4b8565df-94c0-4767-a937-9e914a273674')]

In [45]:
llm_with_tools.invoke(messages).content

'The result of multiplying 2 by 10 is 20.'